In [0]:
df = spark.read.csv(
    "/Workspace/Users/peeyushraj1010@gmail.com/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
df_clean = df.dropDuplicates()

In [0]:
df_clean = df_clean.na.fill("Unknown")

In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|         0|     0|            0|      0|         0|     0|           0|            0|              0|             0|           0|               0|          0|          0|              0|       0|               0| 

In [0]:
df_clean.filter(
    df_clean.Contract == "Month-to-month"
).show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|9763-GRSKD|  Male|            0|    Yes|       Yes|    13|  

In [0]:
from pyspark.sql.functions import *

df_clean.agg(
    count("*").alias("Total_Records"),
    avg("MonthlyCharges").alias("Average_Charges"),
    min("MonthlyCharges").alias("Minimum_Charges"),
    max("MonthlyCharges").alias("Maximum_Charges")
).show()

+-------------+-----------------+---------------+---------------+
|Total_Records|  Average_Charges|Minimum_Charges|Maximum_Charges|
+-------------+-----------------+---------------+---------------+
|         7043|64.76169246059919|          18.25|         118.75|
+-------------+-----------------+---------------+---------------+



In [0]:
df_clean.groupBy("InternetService") \
        .count() \
        .show()

+---------------+-----+
|InternetService|count|
+---------------+-----+
|            DSL| 2421|
|             No| 1526|
|    Fiber optic| 3096|
+---------------+-----+



In [0]:
df_clean.groupBy("Contract") \
        .agg(
            avg("MonthlyCharges").alias("Avg_Charges")
        ) \
        .show()

+--------------+-----------------+
|      Contract|      Avg_Charges|
+--------------+-----------------+
|Month-to-month|66.39849032258043|
|      One year|65.04860828241695|
|      Two year|60.77041297935101|
+--------------+-----------------+



In [0]:
df_new = df_clean.withColumnRenamed(
    "MonthlyCharges",
    "Monthly_Charges"
)

In [0]:
from pyspark.sql.functions import col

df_new = df_new.withColumn(
    "Monthly_Charges",
    col("Monthly_Charges").cast("double")
)

In [0]:
from pyspark.sql.functions import *

final_df = (
    df.dropDuplicates()
      .na.fill("Unknown")
      .groupBy("Contract")
      .agg(
          avg("MonthlyCharges").alias("Average_Charges")
      )
)

final_df.show()

+--------------+-----------------+
|      Contract|  Average_Charges|
+--------------+-----------------+
|Month-to-month|66.39849032258043|
|      One year|65.04860828241695|
|      Two year|60.77041297935101|
+--------------+-----------------+



In [0]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
df_no_duplicates = df.dropDuplicates()
df_no_duplicates.count()

7043

In [0]:
df.filter(df.Contract == "Month-to-month") \
  .groupBy("InternetService") \
  .avg("MonthlyCharges") \
  .show()

+---------------+-------------------+
|InternetService|avg(MonthlyCharges)|
+---------------+-------------------+
|    Fiber optic|  87.02119360902253|
|            DSL|  50.21950122649233|
|             No| 20.409541984732797|
+---------------+-------------------+



In [0]:
df.na.fill("Unknown").show(5)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
from pyspark.sql.functions import count

df.groupBy("InternetService") \
  .agg(count("*").alias("Total_Customers")) \
  .filter("Total_Customers > 100") \
  .show()

+---------------+---------------+
|InternetService|Total_Customers|
+---------------+---------------+
|    Fiber optic|           3096|
|            DSL|           2421|
|             No|           1526|
+---------------+---------------+



In [0]:
df.filter(
    (df.tenure >= 18) &
    (df.tenure <= 30) &
    (df.Contract == "Month-to-month")
).show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|1452-KIOVK|  Male|            0|     No|       Yes|    22|         Yes|             Yes|    Fiber optic|            No|         Yes|              No|         No|    

In [0]:
from pyspark.sql.functions import col

df_new = df.withColumn(
    "TotalChargesDouble",
    col("TotalCharges").cast("double")
)

df_new.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)
 |-- TotalChargesDouble: double (nullable = true)



In [0]:
df_clean = df.filter(
    (df.TotalCharges.isNotNull()) &
    (df.gender != "")
)

df_clean.show(5)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("MonthlyCharges").alias("Min"),
    max("MonthlyCharges").alias("Max"),
    avg("MonthlyCharges").alias("Average")
).show()

+-----+------+-----------------+
|  Min|   Max|          Average|
+-----+------+-----------------+
|18.25|118.75|64.76169246059915|
+-----+------+-----------------+

